In [ ]:
import numpy as np
import sympy as smp
import matplotlib.pyplot as plt
import pandas as pd
from scipy.interpolate import RectBivariateSpline, interp2d

import logging
# Basic registry settings
logging.basicConfig(level=logging.INFO)

In [ ]:
import Cosmo_util_data as cu
import Cosmo_integration as ci
import Cosmo_shear as cs

Interpolation of pkz-Fiducial.txt done
Interpolation of pkz-Om_pl_eps_1p3E-2.txt done
Interpolation of pkz-Om_mn_eps_1p3E-2.txt done
Interpolation of pkz-h_pl_eps_1p3E-2.txt done
Interpolation of pkz-h_mn_eps_1p3E-2.txt done
Interpolation of pkz-Ob_pl_eps_1p3E-2.txt done
Interpolation of pkz-Ob_mn_eps_1p3E-2.txt done
Interpolation of pkz-ns_pl_eps_1p3E-2.txt done
Interpolation of pkz-ns_mn_eps_1p3E-2.txt done
Interpolation of pkz-s8_pl_eps_1p3E-2.txt done
Interpolation of pkz-s8_mn_eps_1p3E-2.txt done
Interpolation of luminosity function created.


In [ ]:
# Parametros fiduciales

Omega_b0_fid = 0.05
Omega_m0_fid = 0.32
h_fid = 0.67
ns_fid = 0.96
sigma8_fid = 0.816
Omega_DE0_fid = 0.68
w0_fid = -1.0
wa_fid = 0.0
gamma_fid = 0.55

# c = 9.72 * 10 ** (-15) # en Mpc # 300000 en km/s
c = 300000 #en km/s
Aia = 1.72
Cia = 0.0134
nia = -0.41
bia = 2.17


In [ ]:
S_Om_m = 0.018
S_h = 0.21
S_Om_b = 0.47
S_ns = 0.035
S_sig = 0.0087

# Esto no sé
S_Aia = 1
S_nia = 1
S_bia = 1

In [ ]:
class Fisher:
    '''
    Calculate Fisher Matrix
    '''
    def __init__(self, params):
        self.num = params['num_params']
        self.universes = params['type']
        self.model = params['model']
        self.IA = params['IA']
        self.l_min, self.l_max, self.l_len = params['l']['l_min'], params['l']['l_max'], params['l']['l_len']
        self.z_min, self.z_max, self.z_len = params['zs']['z_min'], params['zs']['z_max'], params['zs']['z_len']
        self.epsilon, self.c = params['another_constants']['epsilon'], params['another_constants']['c']
        self.fsky, self.Nz = params['euclid_specifications']['fsky'], params['euclid_specifications']['Nz']
        self.sigma_epsilon, self.n_gal = params['euclid_specifications']['sigma_epsilon'], params['euclid_specifications']['n_gal']

        IA_parametros = ['Aia', 'nia', 'bia']
        IA_parametros_values = [Aia, nia, bia]
        IA_results_euclid = [S_Aia, S_nia, S_bia]
        if self.model == 'ACDM_flat':
            parametros = ["Omega_m0", "h", "Omega_b0", "ns", "sigma8"]
            parametros_values = [Omega_m0_fid, h_fid, Omega_b0_fid, ns_fid, sigma8_fid]
            results_euclid = [S_Om_m, S_h, S_Om_b, S_ns, S_sig]
        elif self.model == 'ACDM_non_flat':
            parametros = ["Omega_m0", "h", "Omega_b0", "ns", "sigma8", 'Omega_DE0']
        elif self.model == 'non_ACDM_flat':
            parametros = ["Omega_m0", "h", "Omega_b0", "ns", "sigma8", 'w0', 'wa']
        elif self.model == 'non_ACDM_non_flat':
            parametros = ["Omega_m0", "h", "Omega_b0", "ns", "sigma8", 'Omega_DE0', 'w0', 'wa']
        elif self.model == 'non_ACDM_flat_gamma':
            parametros = ["Omega_m0", "h", "Omega_b0", "ns", "sigma8", 'w0', 'wa', 'gamma']       
        elif self.model == 'non_ACDM_non_flat_gamma':
            parametros = ["Omega_m0", "h", "Omega_b0", "ns", "sigma8", 'Omega_DE0', 'w0', 'wa', 'gamma']       

        if self.IA == True:
            parametros = parametros + IA_parametros
            parametros_values = parametros_values + IA_parametros_values
            results_euclid = results_euclid + IA_results_euclid
        
        self.parametros = parametros
        self.parametros_values = parametros_values
        self.results_euclid = results_euclid

    def trace(self, Omega_m0, h, Omega_b0, Omega_DE0, w0, wa, ns, sigma8, gamma, Aia, nia, bia):
        if self.IA == True:
            F = np.zeros((self.num + 3, self.num + 3))
        else:
            F = np.zeros((self.num, self.num))

        L_array = np.log10(np.logspace(np.log10(self.l_min), np.log10(self.l_max), self.l_len))
        zs = np.linspace(self.z_min, self.z_max, self.z_len) #np.logspace(np.log10(self.z_min), np.log10(self.z_max), self.z_len)
        cosmic_parametros = {'l': L_array , 'z': zs, 'type': self.universes, 'model': self.model, 'IA': self.IA, 'epsilon': self.epsilon, 
                             'Nz': self.Nz, 'c': self.c, 'sigma_epsilon': self.sigma_epsilon, 'n_gal': self.n_gal}

        A = cs.CosmicShear(cosmic_parametros)  

        def derivative(i, j, param):
            deriv = A.Der_C_parametro(i ,j, Omega_m0, h, Omega_b0, Omega_DE0, w0, wa, ns, sigma8, gamma, Aia, nia, bia, param)
            return deriv
        
        def Cosmic_Shear(i, j):
            CS = A.Cosmic_Shear(i, j, Omega_m0, h, Omega_b0, Omega_DE0, w0, wa, ns, sigma8, gamma, Aia, nia, bia)
            return CS

        # Crear un diccionario con todas las matrices
        C = np.zeros((self.Nz, self.Nz), dtype=object)
        dC_dq_matrices = {p: np.zeros((self.Nz, self.Nz), dtype=object) for p in self.parametros}
        
        for i in range(self.Nz):
            for j in range(i, self.Nz):  # solo parte triangular superior
                logging.info(f"Calculated pair: {(i, j)}")
                C[i, j] = C[j, i] = Cosmic_Shear(i, j)
                for p in self.parametros:
                    val = derivative(i, j, p)
                    dC_dq_matrices[p][i, j] = val
                    dC_dq_matrices[p][j, i] = val
        
        C_matrix = np.array(C.tolist(), dtype=float)
        C_matrix_2d = np.squeeze(C_matrix)

        dC_dq_matrices_2d = {}
        for p in self.parametros:
            mat = np.array(dC_dq_matrices[p].tolist(), dtype=float)
            dC_dq_matrices_2d[p] = np.squeeze(mat)

        for i, l in enumerate(L_array):

            # definiciones de lambda_k y delta_l como ya las tienes
            def lambda_k(i): 
                lambda_min = np.log10(10**L_array[0])
                lambda_max = np.log10(10**L_array[-1])
                delta_lambda = (lambda_max - lambda_min) / len(L_array)
                return lambda_min + (i - 1)*delta_lambda
            
            delta_l = 10**lambda_k(i + 1) - 10**lambda_k(i)

            L_parameter_new = ((2*(10 ** l) + 1) * delta_l * self.fsky) / 2 
            
            Ci = C_matrix_2d[:, :, i]
            C_inv  = np.linalg.inv(Ci)

            coeficientes = {}
            # Loop automático sobre pares de parámetros
            for a, p in enumerate(self.parametros):
                for b, q in enumerate(self.parametros[a:], start=a):
                    mat_p = dC_dq_matrices_2d[p][:, :, i]
                    mat_q = dC_dq_matrices_2d[q][:, :, i]
                    val   = np.trace(C_inv @ mat_p @ C_inv @ mat_q)
                    coeficientes[(a, b)] = val  

            # Rellena la Fisher matrix
            for (i, j), coef in coeficientes.items():
                F[i, j] += (L_parameter_new * coef)

        # Simetriza
        Fisher = F + F.T - np.diag(F.diagonal())

        logging.info(f"Fisher Matrix: {Fisher}")
        return Fisher
    
    def Covarianzas(self, F):
        Cov = np.linalg.inv(F)
        return Cov
    
    def results(self, Cov):
        def comparison(created, expected):
            return 100*np.abs(1 - (created/expected))
        errors = {}
        errors_relative = {}
        per = {}
        for i, p in enumerate(self.parametros):
            errors[p] = str(np.sqrt(Cov[i, i]))
            errors_relative[p] = (np.sqrt(Cov[i, i]) / self.parametros_values[i])
            per[p] = comparison(errors_relative[p], self.results_euclid[i])
        return errors_relative, per

In [ ]:
params = {'num_params': 5, 'type': 'standard', 'model' : 'ACDM_flat', 'IA': True, 'l': {'l_min': 10, 'l_max': 1500, 'l_len': 100}, 
          'zs': {'z_min': 0.001, 'z_max': 2.5, 'z_len': 10}, 'another_constants': {'epsilon' : 0.013, 'c' : 3e5}, 
          'euclid_specifications': {'fsky' : 0.3636, 'Nz': 10, 'sigma_epsilon': 0.3, 'n_gal' : 30}} #n_gal en arcmin^-2

A = Fisher(params)

In [ ]:
sigma_epsilon = 0.3
n_gal = 30 #arcmin^-2

In [ ]:
F = A.trace(Omega_m0_fid, h_fid, Omega_b0_fid, Omega_DE0_fid, w0_fid, wa_fid, ns_fid, sigma8_fid, gamma_fid, Aia, nia, bia)

INFO:root:Calculated pair: (0, 0)
INFO:root:Calculated pair: (0, 1)
INFO:root:Calculated pair: (0, 2)
INFO:root:Calculated pair: (0, 3)
INFO:root:Calculated pair: (0, 4)
INFO:root:Calculated pair: (0, 5)
INFO:root:Calculated pair: (0, 6)
INFO:root:Calculated pair: (0, 7)
INFO:root:Calculated pair: (0, 8)
INFO:root:Calculated pair: (0, 9)
INFO:root:Calculated pair: (1, 1)
INFO:root:Calculated pair: (1, 2)
INFO:root:Calculated pair: (1, 3)
INFO:root:Calculated pair: (1, 4)
INFO:root:Calculated pair: (1, 5)
INFO:root:Calculated pair: (1, 6)
INFO:root:Calculated pair: (1, 7)
INFO:root:Calculated pair: (1, 8)
INFO:root:Calculated pair: (1, 9)
INFO:root:Calculated pair: (2, 2)
INFO:root:Calculated pair: (2, 3)
INFO:root:Calculated pair: (2, 4)
INFO:root:Calculated pair: (2, 5)
INFO:root:Calculated pair: (2, 6)
INFO:root:Calculated pair: (2, 7)
INFO:root:Calculated pair: (2, 8)
INFO:root:Calculated pair: (2, 9)
INFO:root:Calculated pair: (3, 3)
INFO:root:Calculated pair: (3, 4)
INFO:root:Calc

In [ ]:
Cov = A.Covarianzas(F)

In [ ]:
Cov

array([[ 1.53322713e-06, -8.39273744e-07,  6.93993276e-08,
        -3.19574181e-06, -2.86379177e-06, -2.90005192e-02,
         1.34157482e-02, -8.11663284e-03],
       [-8.39273744e-07,  7.21563841e-07, -7.13360455e-08,
         1.34263512e-06,  2.72134912e-06,  1.29365439e-02,
        -5.46328875e-03,  1.88854701e-03],
       [ 6.93993276e-08, -7.13360455e-08,  1.00146065e-07,
         2.83008617e-07,  1.98553897e-07,  1.31352508e-02,
        -8.39326594e-03,  2.83091761e-03],
       [-3.19574181e-06,  1.34263512e-06,  2.83008617e-07,
         1.15714474e-05, -1.59813271e-06,  1.43948733e-01,
        -9.62486899e-02,  3.79823580e-02],
       [-2.86379177e-06,  2.72134912e-06,  1.98553897e-07,
        -1.59813271e-06,  3.59762117e-05, -9.23285393e-02,
         1.16845464e-01, -3.22847242e-02],
       [-2.90005192e-02,  1.29365439e-02,  1.31352508e-02,
         1.43948734e-01, -9.23285393e-02,  1.32250140e+05,
        -8.60113586e+04,  3.11460939e+04],
       [ 1.34157482e-02, -5.463288

In [ ]:
rel, perct = A.results(Cov)

In [ ]:
rel

{'Omega_m0': np.float64(0.0038694859062834356),
 'h': np.float64(0.0012678345370288387),
 'Omega_b0': np.float64(0.0063291726089483675),
 'ns': np.float64(0.003543419820402686),
 'sigma8': np.float64(0.00735051140936803),
 'Aia': np.float64(211.431467243932),
 'nia': np.float64(-577.4411624722063),
 'bia': np.float64(39.521545035867994)}

In [ ]:
perct

{'Omega_m0': np.float64(78.50285607620313),
 'h': np.float64(99.39626926808151),
 'Omega_b0': np.float64(98.65336753001098),
 'ns': np.float64(89.87594337027804),
 'sigma8': np.float64(15.511363110712296),
 'Aia': np.float64(21043.1467243932),
 'nia': np.float64(57844.116247220634),
 'bia': np.float64(3852.1545035867994)}

In [ ]:
#Con normalizacion

{'Omega_m0': np.float64(0.005005899963608804),
 'h': np.float64(0.001248654205106214),
 'Omega_b0': np.float64(0.018459575106129247),
 'ns': np.float64(0.0035969432211091076),
 'sigma8': np.float64(0.008443202128281964),
 'Aia': np.float64(42.5444761031612),
 'nia': np.float64(-112.7001449718042),
 'bia': np.float64(8.722090020454557)}

# Sin normalizacion

{'Omega_m0': np.float64(0.0006862816600103079),
 'h': np.float64(0.0003382409342869184),
 'Omega_b0': np.float64(0.0024279972635347655),
 'ns': np.float64(0.0004041128604146491),
 'sigma8': np.float64(7.056934487180305e-05),
 'Aia': np.float64(0.07558649968950613),
 'nia': np.float64(-0.20664905506696646),
 'bia': np.float64(0.014502386264304867)}

# Con normalización y k / h_fid para interpolaciones

{'Omega_m0': np.float64(0.003979639042607673),
 'h': np.float64(0.002028462777634231), 
 'Omega_b0': np.float64(0.017195539472970374),
 'ns': np.float64(0.003733582366734273),
 'sigma8': np.float64(0.00571101428358751),
 'Aia': np.float64(50.86280550963239),
 'nia': np.float64(-135.3541824277623),
 'bia': np.float64(10.544582102726743)}

# Con normalización y k * h_fid para inteprolaciones 

{'Omega_m0': np.float64(0.003330074679038021),
 'h': np.float64(0.0005676272823489522),
 'Omega_b0': np.float64(0.008797047857956392),
 'ns': np.float64(0.0031153113055536396),
 'sigma8': np.float64(0.0027213022868906606),
 'Aia': np.float64(23.231157845197544),
 'nia': np.float64(-61.43447214400714),
 'bia': np.float64(4.8947635072457025)}

# Con normalización y tambien incluyendo +- eps en argumento de k para intepolacion cuando se deriva por h

## No se pudo hacer pq hay problemas con la interpolacion hecha en log

# Con normalización y k * h_fid para inteprolaciones pero con 40 puntos en vez de 20

{'Omega_m0': np.float64(0.002257573345551172),
 'h': np.float64(0.00043805235589220424),
 'Omega_b0': np.float64(0.00629707505348826),
 'ns': np.float64(0.0020063322583912164),
 'sigma8': np.float64(0.0016231363765769218),
 'Aia': np.float64(7.113480420200932),
 'nia': np.float64(-18.288530424305122),
 'bia': np.float64(1.6162414925455353)}

# Lo mismo con k / h_fid

{'Omega_m0': np.float64(0.002503939966722782),
 'h': np.float64(0.0015051469485003034),
 'Omega_b0': np.float64(0.004292567786499258),
 'ns': np.float64(0.001554736339962165),
 'sigma8': np.float64(0.0030093822030210875),
 'Aia': np.float64(14.739835407375889),
 'nia': np.float64(-39.42154246125798),
 'bia': np.float64(3.032049167393405)}

# Lo mismo con k / h

{'Omega_m0': np.float64(0.0032474718426748674),
 'h': np.float64(0.0020763198481980514),
 'Omega_b0': np.float64(0.005203472834854604),
 'ns': np.float64(0.003035942468837147),
 'sigma8': np.float64(0.006279407753223875),
 'Aia': np.float64(193.8681607338671),
 'nia': np.float64(-529.6930779686808)}

# Con P * (h**3)

{'Omega_m0': np.float64(0.0038694859062834356),
 'h': np.float64(0.0012678345370288387),
 'Omega_b0': np.float64(0.0063291726089483675),
 'ns': np.float64(0.003543419820402686),
 'sigma8': np.float64(0.00735051140936803),
 'Aia': np.float64(211.431467243932),
 'nia': np.float64(-577.4411624722063),
 'bia': np.float64(39.521545035867994)}

{'Omega_m0': np.float64(0.0032474718426748674),
 'h': np.float64(0.0020763198481980514),
 'Omega_b0': np.float64(0.005203472834854604),
 'ns': np.float64(0.003035942468837147),
 'sigma8': np.float64(0.006279407753223875),
 'Aia': np.float64(193.8681607338671),
 'nia': np.float64(-529.6930779686808),
 'bia': np.float64(36.37026221742954)}